# 02 — Image-Patch Tensor Graph: Tensor-Native vs Flattened Baseline

**Goal:** Build a graph of image patches where each node is an image-patch
tensor `[C, H, W]`.  Train a tensor-aware TGraphX model and compare it
with a flattened baseline that discards spatial structure.

**Why this matters:** This notebook is the core proof-of-concept for
TGraphX.  Most GNN frameworks force you to flatten `[C,H,W]` node features
into flat vectors, discarding the spatial relationship between channels.
TGraphX preserves that structure throughout message passing.

**TGraphX subsystem:** `ConvMessagePassing`, `build_grid_graph`, `image_to_patches`

**Data:** Synthetic image — no download, no torchvision required.

**Runtime:** < 60 seconds on CPU.

## 1. Setup

In [ ]:
# Optional: uncomment to install in Colab
# !pip install -q tgraphx
import torch
import torch.nn as nn
import torch.nn.functional as F
import tgraphx as tgx
from tgraphx import Graph, ConvMessagePassing, build_grid_graph, image_to_patches, patch_grid_shape
print("TGraphX version:", tgx.__version__)

## 2. Scenario

We synthesize a small `[3, 12, 12]` image with a horizontal gradient
(channel 0), vertical gradient (channel 1), and center bump (channel 2).

The image is split into 9 non-overlapping `4×4` patches.  Each patch becomes
a graph node with features `[3, 4, 4]`.  We connect neighboring patches in
a grid graph and train a simple node classifier.

**Key question:** does preserving the `[3,4,4]` tensor shape help compared
with flattening each patch to a `48`-dimensional vector?

## 3. Build Synthetic Image and Patch Graph

In [ ]:
torch.manual_seed(42)
C, H, W = 3, 12, 12
patch_size, stride = 4, 4

# Synthetic image: gradient + center bump → spatial structure matters.
image = torch.randn(C, H, W) * 0.3
yy = torch.linspace(-1, 1, H).view(H,1).expand(H,W)
xx = torch.linspace(-1, 1, W).view(1,W).expand(H,W)
image[0] += yy; image[1] += xx
image[2] += torch.exp(-(xx**2 + yy**2))

# Patchify: image_to_patches expects [B, C, H, W].
patches = image_to_patches(image.unsqueeze(0), patch_size=patch_size, stride=stride)
patches = patches.squeeze(0)          # [N, C, ph, pw]
N, Cp, ph, pw = patches.shape
print(f"Patches: {N} nodes, each shape [{Cp}, {ph}, {pw}]")

# Per-patch label: is mean intensity in channel 0 positive? (encodes horizontal gradient)
y = (patches[:, 0].mean(dim=(-1,-2)) > 0).long()
num_classes = 2

# Build 4-connected grid graph.
grid_h, grid_w = patch_grid_shape(H, W, patch_size, stride)
edge_index = build_grid_graph(grid_h, grid_w, directed=False)
print(f"Grid: {grid_h}×{grid_w} = {N} nodes, {edge_index.size(1)} edges")

# Tensor-native graph: each node feature is [C, ph, pw].
g_tensor = Graph(node_features=patches, edge_index=edge_index, y=y)
# Flattened graph: each node feature is a 1-D vector.
g_flat = Graph(node_features=patches.flatten(1), edge_index=edge_index, y=y)
print(f"\nTensor node shape: {g_tensor.node_features.shape[1:]}")
print(f"Flat   node shape: {g_flat.node_features.shape[1:]}")

## 4. Define Models

In [ ]:
class TensorModel(nn.Module):
    """Tensor-native model: node features are [C, ph, pw] throughout."""
    def __init__(self):
        super().__init__()
        # ConvMessagePassing operates on [C,H,W] tensors — no flatten.
        self.conv = ConvMessagePassing(
            in_shape=(Cp, ph, pw), out_shape=(8, ph, pw)
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))   # [N,8,1,1]
        self.head = nn.Linear(8, num_classes)

    def forward(self, x, edge_index):
        z = self.conv(x, edge_index).relu()        # [N, 8, ph, pw]
        return self.head(self.pool(z).flatten(1))  # [N, num_classes]

class FlatModel(nn.Module):
    """Flattened baseline: node features lose spatial layout."""
    def __init__(self, in_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 32)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x, edge_index):
        # Simple mean-neighbor aggregation (no spatial structure).
        src, dst = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, x[src])
        counts = torch.zeros(x.size(0)).index_add(0, dst, torch.ones(src.size(0)))
        agg = agg / counts.clamp(min=1).unsqueeze(-1)
        return self.fc2(F.relu(self.fc1(agg)))

tensor_model = TensorModel()
flat_model   = FlatModel(patches.flatten(1).size(1))
tp = sum(p.numel() for p in tensor_model.parameters())
fp = sum(p.numel() for p in flat_model.parameters())
print(f"Tensor model parameters: {tp}")
print(f"Flat   model parameters: {fp}")

## 5. Train Both Models

In [ ]:
import time
opt_t = torch.optim.Adam(tensor_model.parameters(), lr=1e-2)
opt_f = torch.optim.Adam(flat_model.parameters(),   lr=1e-2)
EPOCHS = 10

t0 = time.time()
for ep in range(1, EPOCHS+1):
    # Tensor model
    z_t = tensor_model(g_tensor.node_features, g_tensor.edge_index)
    loss_t = F.cross_entropy(z_t, g_tensor.node_labels)
    opt_t.zero_grad(); loss_t.backward(); opt_t.step()
    acc_t = (z_t.detach().argmax(-1) == g_tensor.node_labels).float().mean()

    # Flat baseline
    z_f = flat_model(g_flat.node_features, g_flat.edge_index)
    loss_f = F.cross_entropy(z_f, g_flat.node_labels)
    opt_f.zero_grad(); loss_f.backward(); opt_f.step()
    acc_f = (z_f.detach().argmax(-1) == g_flat.node_labels).float().mean()

    if ep == 1 or ep % 5 == 0:
        print(f"Ep {ep:2d} | tensor loss={loss_t:.4f} acc={acc_t:.3f} | "
              f"flat loss={loss_f:.4f} acc={acc_f:.3f}")

print(f"\nTotal time: {time.time()-t0:.1f}s")

## 6. Verify Gradient Flow and Tensor Shape Preservation

In [ ]:
# Tensor model: verify spatial shape preserved through the whole graph.
z = tensor_model.conv(g_tensor.node_features, g_tensor.edge_index)
print("After ConvMessagePassing — node feature shape:", z.shape)
# Expected: [N, 8, ph, pw] — same spatial dims, more channels.
assert z.shape == (N, 8, ph, pw), f"Unexpected shape: {z.shape}"
print("✓ Spatial dimensions preserved: no silent flattening inside TGraphX.")

# Check gradient is finite and nonzero.
loss_check = tensor_model(g_tensor.node_features, g_tensor.edge_index).sum()
loss_check.backward()
for name, p in tensor_model.named_parameters():
    if p.grad is not None:
        assert torch.isfinite(p.grad).all(), f"Non-finite grad in {name}"
        assert p.grad.abs().sum() > 0, f"Zero grad in {name}"
print("✓ Gradients are finite and nonzero.")

## 7. Key Takeaways

| | Tensor-native model | Flattened baseline |
|---|---|---|
| Node features | `[C, H, W]` preserved | flattened to `D` |
| Message passing | 1×1 conv (spatial-aware) | linear projection |
| Spatial invariants | respected | discarded |
| TGraphX forced? | no — flattening is always opt-in | n/a |

**TGraphX does not force spatial structure.**  You can always use flat
features.  But if your node features *have* spatial structure (patches,
feature maps, spectrograms, volumetric tensors…), TGraphX keeps it intact
so your model can leverage it.

**This demo uses a tiny graph.**  For larger graphs, combine `ConvMessagePassing`
with `NeighborLoader` — see `tutorials/tensor_node_classification_neighbor_loader.py`.

## 8. Next Steps

- **Tutorial:** `tutorials/tensor_node_classification_neighbor_loader.py`
- **Benchmark:** `benchmarks/tensor_vs_flatten_benchmark.py`
- **Limitations:** on this tiny 9-node graph, the two models converge to
  similar accuracy.  The spatial advantage grows with more complex patterns.